# W1 — LiTS Data Pipeline (Kaggle)

**Trước khi chạy:** Settings → *Internet: ON*, *Accelerator: None (CPU)*.

**Add Data → attach CẢ HAI dataset** (LiTS bị tách đôi):
- `andrewmvd/liver-tumor-segmentation` — 131 segmentations + volume 0–50
- `andrewmvd/liver-tumor-segmentation-part-2` — volume 51–130

Luồng: verify (131 cặp) → build manifest+cache → audit (chốt τ) → split → leakage test → Save Version tạo dataset `lits-processed`.

In [ ]:
# 1) Cài thư viện còn thiếu (Kaggle đã có numpy/pandas/sklearn/matplotlib/cv2)
!pip install -q nibabel pyyaml

In [ ]:
# 2) Nạp code từ GitHub + set DATA_ROOT
import os
if not os.path.exists('HCC-TACE-Assist'):
    !git clone -q https://github.com/hdtruong802/HCC-TACE-Assist.git
%cd HCC-TACE-Assist
import sys; sys.path.insert(0, '.')
# /kaggle/input là gốc chứa CẢ HAI dataset (part1 + part2); glob đệ quy tự khớp volume↔segmentation theo id
os.environ['DATA_ROOT'] = '/kaggle/input'
OUT = '/kaggle/working'
print('DATA_ROOT =', os.environ['DATA_ROOT'])

In [ ]:
# 3) VERIFY: đếm cặp volume/seg + đọc thử 1 cặp
from src.data.io import discover_pairs, load_pair
import numpy as np
pairs = discover_pairs(os.environ['DATA_ROOT'])
print('So cap volume/seg:', len(pairs), '(ky vong 131)')
assert len(pairs) > 0, 'Khong tim thay data — kiem tra Add Data / DATA_ROOT'
vol, seg, sp = load_pair(pairs[0])
print(pairs[0].patient_id, 'vol', vol.shape, 'spacing', sp, 'nhan seg:', np.unique(seg))

In [ ]:
# 4) BUILD MANIFEST + CACHE
#    Lần đầu để LIMIT='--limit 3' (smoke ~1 phút). OK rồi đổi LIMIT='' để chạy full 131 vol.
import os
LIMIT = '--limit 3'   # <-- đổi thành '' để chạy FULL
cmd = f"python scripts/build_manifest.py --data-root {os.environ['DATA_ROOT']} --out-dir /kaggle/working {LIMIT}"
print('>', cmd)
!{cmd}

In [ ]:
# 5) AUDIT: phan bo lop + histogram tumor_area (de CHOT tau_area) + QC montage
import pandas as pd
from src.data.audit import class_distribution, tumor_area_hist, qc_montage, write_report
df = pd.read_csv('/kaggle/working/manifest.csv')
dist = class_distribution(df); print(dist)
hist = tumor_area_hist(df, out_png='/kaggle/working/reports/tumor_area_hist.png'); print(hist)
qc = qc_montage(df, cache_root='/kaggle/working', out_png='/kaggle/working/reports/qc_montage.png')
write_report(dist, hist, out_md='/kaggle/working/reports/data_audit_report.md')
from IPython.display import Image, display
display(Image('/kaggle/working/reports/tumor_area_hist.png'))
display(Image('/kaggle/working/reports/qc_montage.png'))

**CHỐT τ_area:** xem `percentiles` + histogram ở trên. Nếu hợp lý giữ mặc định `tau_area=10`; nếu muốn đổi → sửa `configs/data/lits.yaml` rồi chạy lại cell 4 (chỉ gán lại nhãn, cache không đổi).

In [ ]:
# 6) SPLIT patient-level + 7) LEAKAGE TEST
!python scripts/make_split.py --manifest /kaggle/working/manifest.csv --out /kaggle/working/splits/lits_v1.json
!cp -r /kaggle/working/splits ./splits 2>/dev/null; python -m pytest tests/test_leakage.py -q

## 8) Lưu kết quả
1. **Save Version** (Save & Run All) → *Output* của notebook chứa `processed_cache/`, `manifest.csv`, `splits/`, `reports/`.
2. Tạo Dataset từ Output: *Output → New Dataset* đặt tên **`lits-processed`** (private) → W2 attach để train.
3. Tải về máy (nhỏ): `manifest.csv`, `splits/lits_v1.json`, `reports/data_audit_report.md` → commit vào git (KHÔNG commit cache).